# 🎙️ OmniVoice Colab (Localtunnel + Safe Clone)
---
**Features:**
*   Downloads both the custom `omnivoice-colab` repo AND the official `OmniVoice` repo.
*   Includes strict safety checks to prevent `Errno 2` crashes if a download fails.
*   Uses **Localtunnel** to bypass Cloudflare Colab bans.

In [ ]:
#@title 1. Install OmniVoice & Dependencies
import os, sys
from IPython.display import clear_output
import torch

print("🧹 Cleaning old files...")
!rm -rf /content/omnivoice-colab /content/OmniVoice

print("⬇️ Downloading Custom Repository...")
%cd /content/
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git

# SAFEGUARD: Check if the first repo downloaded successfully
if not os.path.exists('/content/omnivoice-colab'):
    print("\n" + "="*60)
    print("❌ DOWNLOAD FAILED: 'omnivoice-colab' folder is missing.")
    print("This usually happens if your GitHub repo is set to PRIVATE or doesn't exist.")
    print("Colab cannot download private repos without an access token.")
    print("="*60 + "\n")
    sys.exit("Execution stopped to prevent Errno 2 crash.")

print("\n⬇️ Downloading Official OmniVoice Repository...")
%cd /content/omnivoice-colab
!git clone https://github.com/k2-fsa/OmniVoice.git

if not os.path.exists('/content/omnivoice-colab/OmniVoice'):
    sys.exit("❌ ERROR: Failed to download the official OmniVoice repo.")

print("\n📦 Installing System Dependencies (FFmpeg)...")
!apt-get update -qq && apt-get install -y ffmpeg

print("\n📦 Installing Python Requirements...")
!pip install -q -r colab.txt
!pip install -q fastapi "uvicorn[standard]" python-multipart httpx requests aiofiles nest-asyncio python-dotenv pydub soundfile librosa

print("\n⬇️ Fetching latest app.py...")
!wget -q -O app.py https://raw.githubusercontent.com/hahyt6644-gif/omnivoice-colab/main/app.py

clear_output()

print("========== SYSTEM INFO ==========")
if torch.cuda.is_available():
    print("✅ GPU ENABLED:", torch.cuda.get_device_name(0))
    print("✅ OmniVoice Installed Successfully")
else:
    print("\n" + "="*50)
    print("❌ CRITICAL ERROR: GPU NOT ENABLED!")
    print("="*50)
    print("OmniVoice requires a GPU to run. Please fix this by:")
    print("1. Clicking 'Runtime' in the top menu.")
    print("2. Clicking 'Change runtime type'.")
    print("3. Selecting 'T4 GPU' under Hardware accelerator.")
    print("4. Clicking 'Save' and running this cell again.")
    print("="*50 + "\n")
    sys.exit("Execution stopped: GPU required to continue.")

In [ ]:
#@title 2. Run Server & Generate Localtunnel URL
%cd /content/omnivoice-colab

# Kill previous server and tunnels to free up ports
!pkill -9 -f "python app.py" || true
!pkill -9 -f "localtunnel" || true
!pkill -9 -f "node" || true

# Install Localtunnel globally via npm
!npm install -g localtunnel >/dev/null 2>&1

import subprocess, time, re, requests, os, threading
from IPython.display import clear_output

print("🚀 Starting OmniVoice Server...")
server = subprocess.Popen(['python', 'app.py'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

server_ready = False
for i in range(30):
    try:
        requests.get('http://127.0.0.1:7860', timeout=2)
        server_ready = True
        break
    except:
        time.sleep(2)

if not server_ready:
    print("❌ Server failed to start. Printing crash logs:\n")
    os.system("tail -n 30 server.log")
else:
    print("\n🌍 Starting Localtunnel...\n")
    tunnel = subprocess.Popen(
        ["lt", "--port", "7860"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    url = None
    start = time.time()
    while time.time() - start < 60:
        line = tunnel.stdout.readline()
        if not line: continue
        match = re.search(r'https?://[a-zA-Z0-9.-]+\.loca\.lt', line)
        if match:
            url = match.group(0)
            break

    clear_output()

    if url:
        try:
            endpoint_ip = requests.get('https://ipv4.icanhazip.com').text.strip()
        except:
            endpoint_ip = "Couldn't fetch IP. Check your Colab public IP."
            
        print("=" * 60)
        print("✅ OMNIVOICE RUNNING VIA LOCALTUNNEL")
        print("=" * 60)
        print(f"\n🌍 Public URL:\n{url}")
        print(f"\n🖥 UI:\n{url}/ui")
        print(f"\n🎙 Clone API:\n{url}/api/clone")
        print(f"\n🎨 Design API:\n{url}/api/design")
        print("\n" + "-" * 60)
        print("🔒 LOCALTUNNEL SECURITY BYPASS:")
        print("When you first click the link, it will ask for a password.")
        print(f"Copy and paste this IP address: {endpoint_ip}")
        print("-" * 60 + "\n")
        print("⚠️ Keep notebook running. Restart cell = New URL")
        print("=" * 60)

        def consume_tunnel_output(tunnel_proc):
            for _ in tunnel_proc.stdout: pass
        threading.Thread(target=consume_tunnel_output, args=(tunnel,), daemon=True).start()
    else:
        print("❌ Failed to get Localtunnel URL. Try running the cell again.")